In [1]:
import pandas as pd
from sklearn.metrics import root_mean_squared_error
import pickle as pkl
import numpy as np

In [3]:
# load in the test data
with open('../data/processed/df_tabular_test.pkl', 'rb') as f:

    df = pkl.load(f)

In [4]:
# drop columns that are not lag 1 hour or 1 week
cols_to_drop = df.columns.difference(set(('lag_1hr', 'lag_1wk', 'hourly_usage_kwh')))
df.drop(columns=cols_to_drop, inplace=True)

In [5]:
def unscale_per_client(group, dict_with_mean_and_std):
    """ use mean and std usage for each client to unscale data, ready for 
    evaluation """

    client_id = int(np.unique(group.index.get_level_values('client_id'))[0])

    return ((group * dict_with_mean_and_std[client_id]['std']) + dict_with_mean_and_std[client_id]['mean'])

In [6]:
# load in mean and std usages for converting labels and predictions back to original units and for nrmse
# computation
with open('../data/processed/mean_std_per_client.json', 'r') as f:
    df_std_mean_usage = pd.read_json(f).T
    dict_std_mean_usage = df_std_mean_usage.to_dict(orient='index')

In [7]:
# create new feature which is the average usage at that hour and day of week for all time

# now unscale the predictions to get original units
df_unscaled = df.groupby(level='client_id').transform(lambda g: unscale_per_client(g, dict_std_mean_usage))

In [16]:
# compute the rmse for each client
client_rmse_1hr_lag = df_unscaled.groupby(level='client_id').apply(lambda g: root_mean_squared_error(g['hourly_usage_kwh'], g['lag_1hr']))
client_rmse_1wk_lag = df_unscaled.groupby(level='client_id').apply(lambda g: root_mean_squared_error(g['hourly_usage_kwh'], g['lag_1wk']))

In [17]:
#TODO: nrmse too small - a bug somewhere
client_nrmse_1hr_lag = client_rmse_1hr_lag/df_std_mean_usage['mean']
client_nrmse_1wk_lag = client_rmse_1wk_lag/df_std_mean_usage['mean']

In [18]:
# compute summary stats
summary_dict_1hr_lag = {
                'mean': client_nrmse_1hr_lag.mean(),
                'std': client_nrmse_1hr_lag.std(),
                'max': client_nrmse_1hr_lag.max(),
                'min': client_nrmse_1hr_lag.min(),
                }

summary_dict_1wk_lag = {
                'mean': client_nrmse_1wk_lag.mean(),
                'std': client_nrmse_1wk_lag.std(),
                'max': client_nrmse_1wk_lag.max(),
                'min': client_nrmse_1wk_lag.min(),
                }

In [19]:
import json

with open('../logs/naive/nrmse_summary_stats_1hr_lag.json', 'w') as f:
    json.dump(summary_dict_1hr_lag, f, indent=4)

with open('../logs/naive/nrmse_summary_stats_1wk_lag.json', 'w') as f:
    json.dump(summary_dict_1wk_lag, f, indent=4)

with open('../logs/naive/nrmse_per_client_1hr_lag.json', 'w') as f:
    json.dump(client_nrmse_1hr_lag.to_list(), f, indent=4)

with open('../logs/naive/nrmse_per_client_1wk_lag.json', 'w') as f:
    json.dump(client_nrmse_1wk_lag.to_list(), f, indent=4)